In [2]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt

In [3]:
# input array = [t,\zeta^0, \zeta^1, \xi^0, \xi^1]
# t = array[0]
# zeta_01 = array[1][0]
# zeta_02 = array[1][1]
# zeta_03 = array[1][2]
#
# zeta_11 = array[2][0]
# zeta_12 = array[2][1]
# zeta_13 = array[2][2]
# 
# xi_01 = array[3][0]
# xi_02 = array[3][1]
# xi_03 = array[3][2]
#
# xi_11 = array[3][0]
# xi_12 = array[3][1]
# xi_13 = array[3][2]


In [4]:
# input array = [t,u, v]
# t = array[0]

# u_1 = array[1][0]
# u_2 = array[1][1]

# v_1 = array[2][0]
# v_2 = array[2][1]


In [5]:
# ---------------------------------------------------------------------------
# Parameters -- set these to your problem's values
# ---------------------------------------------------------------------------
n_r = 1.5      # refractive index, must satisfy n_r > 1
rho_0 = 1.0    # rho_0 > 0
d_0 = 1.0      # d_0 > 0
w = np.array([0.0, 1.0])   # fixed vector w = (0, 1)

def _unpack(array):
    """Split the input array into t, u_1, u_2, v_1, v_2."""
    t = array[0]
    u_1 = array[1][0]
    u_2 = array[1][1]
    v_1 = array[2][0]
    v_2 = array[2][1]
    return t, u_1, u_2, v_1, v_2


# ===========================================================================
# First set: auxiliary functions (eq:auxiliary)
# ===========================================================================

def alpha_r(array):
    """alpha_r(u) = (u_1 - sqrt((u_1^2+u_2^2)(n_r^2-1) + u_1^2)) / (u_1^2+u_2^2)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    norm_sq = u_1**2 + u_2**2
    root = np.sqrt(norm_sq * (n_r**2 - 1) + u_1**2)
    return (u_1 - root) / norm_sq


def M_r(array):
    """
    M_r(t,u) = (1/n_r) [ (sin t, cos t)
                         - alpha_r(u) (u_1 sin t - u_2 cos t,
                                     u_1 cos t + u_2 sin t) ].
    Returns np.array([M_r1, M_r2]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_r(array)
    vec1 = np.array([np.sin(t), np.cos(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_1 * np.cos(t) + u_2 * np.sin(t)])
    return (vec1 - a * vec2) / n_r

# def M_r1(array):
#     """
#     M_r(t,u) = (1/n_r) [ (sin t, cos t)
#                          - alpha_r(u) (u_1 sin t - u_2 cos t,
#                                      u_1 cos t + u_2 sin t) ].
#     Returns np.array([M_r1, M_r2]).
#     """
#     t, u_1, u_2, v_1, v_2 = _unpack(array)
#     a = alpha_r(array)
#     return (np.sin(t) - a * (u_1 * np.sin(t) - u_2 * np.cos(t))) / n_r

# def M_r2(array):
#     """
#     M_r(t,u) = (1/n_r) [ (sin t, cos t)
#                          - alpha_r(u) (u_1 sin t - u_2 cos t,
#                                      u_1 cos t + u_2 sin t) ].
#     Returns np.array([M_r1, M_r2]).
#     """
#     t, u_1, u_2, v_1, v_2 = _unpack(array)
#     a = alpha_r(array)
#     return (np.cos(t) - a * ( u_1 * np.cos(t) + u_2 * np.sin(t))) / n_r

def D_r(array):
    """D_r(t,u) = ((n_r-1) d_0 - u_1 (1 - cos t)) / (n_r - w . M_r(t,u))."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mr = M_r(array)
    numerator = (n_r - 1) * d_0 - u_1 * (1 - np.cos(t))
    denominator = n_r - np.dot(w, Mr)
    return numerator / denominator


def F_r(array):
    """
    F_r(t,u) = u_1 (sin t, cos t) + D_r(t,u) M_r(t,u).
    Returns np.array([F_r1, F_r2]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    return u_1 * np.array([np.sin(t), np.cos(t)]) + D_r(array) * M_r(array)


def Lambda_2r(array):
    """Lambda_{2,r}(t,u) = sqrt(1 + 1/n_r^2 - (2/n_r) w . M_r(t,u))."""
    Mr = M_r(array)
    return np.sqrt(1.0 + 1.0 / n_r**2 - (2.0 / n_r) * np.dot(w, Mr))


# ===========================================================================
# Second set: tilde auxiliary functions (eq:aux with tilde)
# ===========================================================================

def alpha_r_tilde(array):
    """
    alpha_r_tilde(u,v) = (n_r^2-1) *
        ( v_1 + ((u_1 v_1 + u_2 v_2)(n_r^2-1) + u_1 v_1) / sqrt((u_1^2+u_2^2)(n_r^2-1) + u_1^2) )
        * ( 1 / (u_1 + sqrt((u_1^2+u_2^2)(n_r^2-1) + u_1^2)) )^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    root = np.sqrt((u_1**2 + u_2**2) * (n_r**2 - 1) + u_1**2)
    middle = v_1 + ((u_1 * v_1 + u_2 * v_2) * (n_r**2 - 1) + u_1 * v_1) / root
    return (n_r**2 - 1) * middle * (1.0 / (u_1 + root))**2


def M_r_tilde(array):
    """
    M_r_tilde(t,u,v) = (1/n_r) [ (cos t, -sin t)
        - alpha_r_tilde(u,v) (u_1 sin t - u_2 cos t,  u_2 sin t + u_1 cos t)
        - alpha_r(u) ((u_1 - v_2) cos t + (u_2 + v_1) sin t,
                    (v_2 - u_1) sin t + (v_1 + u_2) cos t) ].
    Returns np.array([M_r1_tilde, M_r2_tilde]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_r(array)
    a_t = alpha_r_tilde(array)

    vec1 = np.array([np.cos(t), -np.sin(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3 = np.array([(u_1 - v_2) * np.cos(t) + (u_2 + v_1) * np.sin(t),
                     (v_2 - u_1) * np.sin(t) + (v_1 + u_2) * np.cos(t)])

    return (vec1 - a_t * vec2 - a * vec3) / n_r


def D_r_tilde(array):
    """
    D_r_tilde(t,u,v) =
        [ (v_1 (cos t - 1) - u_1 sin t)(n_r - w . M_r)
          + ((n_r-1) d_0 - u_1 (1 - cos t)) (w . M_r_tilde) ]
        / (n_r - w . M_r)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mr = M_r(array)
    Mr_t = M_r_tilde(array)
    w_dot_Mr = np.dot(w, Mr)
    w_dot_Mr_t = np.dot(w, Mr_t)

    numerator = ((v_1 * (np.cos(t) - 1) - u_1 * np.sin(t)) * (n_r - w_dot_Mr)
                 + ((n_r - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Mr_t)
    return numerator / (n_r - w_dot_Mr)**2


def F_r_tilde(array):
    """
    F_r_tilde(t,u,v) = u_1 (cos t, -sin t) + v_1 (sin t, cos t)
                       + M_r_tilde(t,u,v) D_r(t,u) + M_r(t,u) D_r_tilde(t,u,v).
    Returns np.array([F_r1_tilde, F_r2_tilde]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    term1 = u_1 * np.array([np.cos(t), -np.sin(t)])
    term2 = v_1 * np.array([np.sin(t), np.cos(t)])
    return term1 + term2 + M_r_tilde(array) * D_r(array) + M_r(array) * D_r_tilde(array)


def Lambda_2r_tilde(array):
    """Lambda_2r_tilde(t,u,v) = -(1/n_r) (w . M_r_tilde) / Lambda_2r(t,u)."""
    Mr_t = M_r_tilde(array)
    return -(1.0 / n_r) * np.dot(w, Mr_t) / Lambda_2r(array)



# # ===========================================================================
# # Quick sanity check
# # ===========================================================================
# if __name__ == "__main__":
#     # Example input: t = 0.3, u = (1.0, 0.5), v = (0.2, -0.1)
#     test = [0.3, np.array([1.0, 0.5]), np.array([0.2, -0.1])]

#     print("alpha_r          =", alpha_r(test))
#     print("M_r            =", M_r(test))
#     print("D_r            =", D_r(test))
#     print("F_r            =", F_r(test))
#     print("Lambda_2r      =", Lambda_2r(test))
#     print()
#     print("alpha_r_tilde    =", alpha_r_tilde(test))
#     print("M_r_tilde      =", M_r_tilde(test))
#     print("D_r_tilde      =", D_r_tilde(test))
#     print("F_r_tilde      =", F_r_tilde(test))
#     print("Lambda_2r_tilde=", Lambda_2r_tilde(test))

In [6]:
"""
Auxiliary functions for the Picard-Lindelof contraction solver -- wavelength B.

All functions take a single array input with the structure:
    array = [t, u, v]
where
    t   = array[0]          (scalar)
    u_1 = array[1][0]
    u_2 = array[1][1]
    v_1 = array[2][0]
    v_2 = array[2][1]

Vector-valued maps return numpy arrays of shape (2,).

Parameters (n_b > 1, rho_0 > 0, d_0 > 0) are module-level constants;
adjust them below as needed. w = (0, 1) is fixed.
"""

# ---------------------------------------------------------------------------
# Parameters -- set these to your problem's values
# ---------------------------------------------------------------------------
n_b = 1.5      # refractive index for wavelength B, must satisfy n_b > 1
rho_0 = 1.0    # rho_0 > 0
d_0 = 1.0      # d_0 > 0
w = np.array([0.0, 1.0])   # fixed vector w = (0, 1)


def _unpack(array):
    """Split the input array into t, u_1, u_2, v_1, v_2."""
    t = array[0]
    u_1 = array[1][0]
    u_2 = array[1][1]
    v_1 = array[2][0]
    v_2 = array[2][1]
    return t, u_1, u_2, v_1, v_2


# ===========================================================================
# First set: auxiliary functions (eq:auxiliary) -- wavelength B
# ===========================================================================

def alpha_b(array):
    """alpha_b(u) = (u_1 - sqrt((u_1^2+u_2^2)(n_b^2-1) + u_1^2)) / (u_1^2+u_2^2)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    norm_sq = u_1**2 + u_2**2
    root = np.sqrt(norm_sq * (n_b**2 - 1) + u_1**2)
    return (u_1 - root) / norm_sq


def M_b(array):
    """
    M_b(t,u) = (1/n_b) [ (sin t, cos t)
                         - alpha_b(u) (u_1 sin t - u_2 cos t,
                                       u_1 cos t + u_2 sin t) ].
    Returns np.array([M_b1, M_b2]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_b(array)
    vec1 = np.array([np.sin(t), np.cos(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_1 * np.cos(t) + u_2 * np.sin(t)])
    return (vec1 - a * vec2) / n_b


def D_b(array):
    """D_b(t,u) = ((n_b-1) d_0 - u_1 (1 - cos t)) / (n_b - w . M_b(t,u))."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mb = M_b(array)
    numerator = (n_b - 1) * d_0 - u_1 * (1 - np.cos(t))
    denominator = n_b - np.dot(w, Mb)
    return numerator / denominator


def F_b(array):
    """
    F_b(t,u) = u_1 (sin t, cos t) + D_b(t,u) M_b(t,u).
    Returns np.array([F_b1, F_b2]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    return u_1 * np.array([np.sin(t), np.cos(t)]) + D_b(array) * M_b(array)


def Lambda_2b(array):
    """Lambda_{2,b}(t,u) = sqrt(1 + 1/n_b^2 - (2/n_b) w . M_b(t,u))."""
    Mb = M_b(array)
    return np.sqrt(1.0 + 1.0 / n_b**2 - (2.0 / n_b) * np.dot(w, Mb))


# ===========================================================================
# Second set: tilde auxiliary functions (eq:aux with tilde) -- wavelength B
# ===========================================================================

def alpha_tilde_b(array):
    """
    alpha_tilde_b(u,v) = (n_b^2-1) *
        ( v_1 + ((u_1 v_1 + u_2 v_2)(n_b^2-1) + u_1 v_1) / sqrt((u_1^2+u_2^2)(n_b^2-1) + u_1^2) )
        * ( 1 / (u_1 + sqrt((u_1^2+u_2^2)(n_b^2-1) + u_1^2)) )^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    root = np.sqrt((u_1**2 + u_2**2) * (n_b**2 - 1) + u_1**2)
    middle = v_1 + ((u_1 * v_1 + u_2 * v_2) * (n_b**2 - 1) + u_1 * v_1) / root
    return (n_b**2 - 1) * middle * (1.0 / (u_1 + root))**2


def M_b_tilde(array):
    """
    M_b_tilde(t,u,v) = (1/n_b) [ (cos t, -sin t)
        - alpha_tilde_b(u,v) (u_1 sin t - u_2 cos t,  u_2 sin t + u_1 cos t)
        - alpha_b(u) ((u_1 - v_2) cos t + (u_2 + v_1) sin t,
                      (v_2 - u_1) sin t + (v_1 + u_2) cos t) ].
    Returns np.array([M_b1_tilde, M_b2_tilde]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_b(array)
    a_t = alpha_tilde_b(array)

    vec1 = np.array([np.cos(t), -np.sin(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3 = np.array([(u_1 - v_2) * np.cos(t) + (u_2 + v_1) * np.sin(t),
                     (v_2 - u_1) * np.sin(t) + (v_1 + u_2) * np.cos(t)])

    return (vec1 - a_t * vec2 - a * vec3) / n_b


def D_b_tilde(array):
    """
    D_b_tilde(t,u,v) =
        [ (v_1 (cos t - 1) - u_1 sin t)(n_b - w . M_b)
          + ((n_b-1) d_0 - u_1 (1 - cos t)) (w . M_b_tilde) ]
        / (n_b - w . M_b)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mb = M_b(array)
    Mb_t = M_b_tilde(array)
    w_dot_Mb = np.dot(w, Mb)
    w_dot_Mb_t = np.dot(w, Mb_t)

    numerator = ((v_1 * (np.cos(t) - 1) - u_1 * np.sin(t)) * (n_b - w_dot_Mb)
                 + ((n_b - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Mb_t)
    return numerator / (n_b - w_dot_Mb)**2


def F_b_tilde(array):
    """
    F_b_tilde(t,u,v) = u_1 (cos t, -sin t) + v_1 (sin t, cos t)
                       + M_b_tilde(t,u,v) D_b(t,u) + M_b(t,u) D_b_tilde(t,u,v).
    Returns np.array([F_b1_tilde, F_b2_tilde]).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    term1 = u_1 * np.array([np.cos(t), -np.sin(t)])
    term2 = v_1 * np.array([np.sin(t), np.cos(t)])
    return term1 + term2 + M_b_tilde(array) * D_b(array) + M_b(array) * D_b_tilde(array)


def Lambda_2b_tilde(array):
    """Lambda_2b_tilde(t,u,v) = -(1/n_b) (w . M_b_tilde) / Lambda_2b(t,u)."""
    Mb_t = M_b_tilde(array)
    return -(1.0 / n_b) * np.dot(w, Mb_t) / Lambda_2b(array)


# # ===========================================================================
# # Quick sanity check
# # ===========================================================================
# if __name__ == "__main__":
#     # Example input: t = 0.3, u = (1.0, 0.5), v = (0.2, -0.1)
#     test = [0.3, np.array([1.0, 0.5]), np.array([0.2, -0.1])]

#     print("alpha_b          =", alpha_b(test))
#     print("M_b              =", M_b(test))
#     print("D_b              =", D_b(test))
#     print("F_b              =", F_b(test))
#     print("Lambda_2b        =", Lambda_2b(test))
#     print()
#     print("alpha_tilde_b    =", alpha_tilde_b(test))
#     print("M_b_tilde        =", M_b_tilde(test))
#     print("D_b_tilde        =", D_b_tilde(test))
#     print("F_b_tilde        =", F_b_tilde(test))
#     print("Lambda_2b_tilde  =", Lambda_2b_tilde(test))

#     # Cross-check: alpha_tilde_b should match the finite-difference directional
#     # derivative of alpha_b in direction v
#     u = np.array([1.0, 0.5]); v = np.array([0.2, -0.1]); t = 0.3; h = 1e-7
#     fd = (alpha_b([t, u + h*v, v]) - alpha_b([t, u - h*v, v])) / (2*h)
#     print()
#     print("alpha_tilde_b (analytic)   :", alpha_tilde_b(test))
#     print("alpha_tilde_b (finite diff):", fd)